# Step 12.4 — Implement a Single Causal Attention Head：把前面的零件第一次连成 MLX Forward

到 Step 12.3，我们已经分别掌握了：

```text
X -> Q / K / V projections
QK^T compatibility scores
/ sqrt(d_k) scaling
causal visibility j <= i
additive -inf mask
stable softmax over Key axis
A @ V retrieval
```

这一节第一次把它们封装成一个最小 **single causal self-attention head（单头因果自注意力）**。

目标不是继续引入新数学，而是把已经学过的公式准确翻译成 MLX `nn.Module` forward，并把每一步 shape 对齐。

本节不进入 Step 12.5 的专门 attention-matrix inspection，也不进入 Multi-Head Attention。


## 1. 一个 Single Head 需要哪些 Trainable Parts？

输入：

$$
X\in\mathbb R^{B\times T\times C}.
$$

本节用一个 head width：

$$
d_h
$$

并令：

$$
d_k=d_v=d_h
$$

只是为了让第一个实现更紧凑。一般理论上 `d_k` 与 `d_v` 可以不同，Step 11 已经讲过。

一个最小 single head 需要三套 trainable linear projections：

$$
Q=XW_Q,
\qquad
K=XW_K,
\qquad
V=XW_V.
$$

在 MLX 0.32.2 中写成：

```python
self.q_proj = nn.Linear(input_dims=C, output_dims=d_h, bias=False)
self.k_proj = nn.Linear(input_dims=C, output_dims=d_h, bias=False)
self.v_proj = nn.Linear(input_dims=C, output_dims=d_h, bias=False)
```

这些 `Linear.weight` 才是模型需要学习的参数。

Scaling factor：

$$
\frac{1}{\sqrt{d_h}}
$$

只是由维度决定的 Python scalar，不需要训练。


## 2. Causal Mask 为什么不应该是 Trainable Parameter？

Causal rule：

$$
j\le i
$$

来自 autoregressive task definition，不是模型应该从数据里学习出来的东西。

所以 mask 的语义是：

> **non-trainable structural constant / state（非训练结构常量 / 状态）**。

这里有一个 MLX 0.32.2 很重要的实现细节：

`nn.Module` 会把直接挂在 module 上的 `mx.array` attribute 收进 `parameters()`；而 module 中的 arrays 默认都是 trainable，除非显式 `freeze(...)`。

因此如果直接写：

```python
self.causal_mask = some_mx_array
```

却什么都不做，就容易把本来固定的结构 mask 混进 parameter tree。

### 本节采用的方案：Forward 内局部构造

本节为了最清楚和最安全，选择：

```text
根据当前 T
在 __call__ 内构造 mask
↓
只作为 forward 的 local tensor
↓
不存到 self.*
↓
不会进入 model parameter tree
```

这样还天然支持不同 sequence length `T`。

### 如果以后想 Cache 呢？

固定 max context length 时，可以预先缓存 mask，但若把 MLX array 放进 module state，就应该明确 freeze 对应 key，使它不出现在 `trainable_parameters()` 中。

本节不做缓存优化；只建立“mask 是结构状态，不是 learned weight”的原则。


In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn


class CausalSelfAttentionHead(nn.Module):
    def __init__(self, input_dims: int, head_dim: int):
        super().__init__()

        self.q_proj = nn.Linear(
            input_dims=input_dims,
            output_dims=head_dim,
            bias=False,
        )
        self.k_proj = nn.Linear(
            input_dims=input_dims,
            output_dims=head_dim,
            bias=False,
        )
        self.v_proj = nn.Linear(
            input_dims=input_dims,
            output_dims=head_dim,
            bias=False,
        )

        # Python float: structural scale, not a trainable MLX array.
        self.scale = 1.0 / math.sqrt(head_dim)

    def __call__(self, x: mx.array) -> mx.array:
        # x: (B,T,C)
        _, T, _ = x.shape

        # 1) Role-specific projections
        q = self.q_proj(x)  # (B,T,d_h)
        k = self.k_proj(x)  # (B,T,d_h)
        v = self.v_proj(x)  # (B,T,d_h)

        # 2) Pairwise compatibility + scaling
        k_t = mx.swapaxes(k, -1, -2)       # (B,d_h,T)
        scores = (q @ k_t) * self.scale    # (B,T,T)

        # 3) Local causal structural constant for this T
        future_upper = mx.triu(mx.ones((T, T)), k=1) > 0
        neg_inf = mx.array(float('-inf'), dtype=scores.dtype)
        zero = mx.array(0.0, dtype=scores.dtype)
        causal_mask = mx.where(future_upper, neg_inf, zero)  # (T,T)

        # 4) Mask before softmax
        masked_scores = scores + causal_mask  # (B,T,T)

        # 5) Stable softmax across Key positions
        attention_weights = mx.softmax(masked_scores, axis=-1)  # (B,T,T)

        # 6) Weighted Value retrieval
        out = attention_weights @ v  # (B,T,d_h)

        return out


## 3. Forward Shape Trace：每一行代码对应什么数学？

设：

```text
B = batch size
T = sequence length
C = input width
d_h = this head's feature width
```

输入：

$$
X:(B,T,C).
$$

### Projection

```text
Q : (B,T,d_h)
K : (B,T,d_h)
V : (B,T,d_h)
```

### Key transpose

只交换 K 最后两个 axes：

$$
(B,T,d_h)
\rightarrow
(B,d_h,T).
$$

### Compatibility

$$
(B,T,d_h)
@
(B,d_h,T)
\rightarrow
(B,T,T).
$$

再乘：

$$
1/\sqrt{d_h}.
$$

shape 不变。

### Causal mask

local mask：

$$
(T,T).
$$

加到：

$$
(B,T,T)
$$

时通过 broadcasting 得到：

$$
(B,T,T).
$$

### Softmax

`axis=-1` 沿 Key positions：

$$
(B,T,T)
\rightarrow
(B,T,T).
$$

### Value retrieval

$$
(B,T,T)
@
(B,T,d_h)
\rightarrow
\boxed{(B,T,d_h)}.
$$

所以这个 single head 保留 B 和 T，最终把每个 position 的 representation 改写成一个 causal context-dependent `d_h`-dimensional vector。


## 4. 代码实际上就是一条 Causal-Attention 公式

上面的 forward 可以压缩为：

$$
Q=XW_Q,
\qquad
K=XW_K,
\qquad
V=XW_V.
$$

然后：

$$
S
=
\frac{QK^T}{\sqrt{d_h}}.
$$

加入 causal mask：

$$
S_{masked}=S+M,
$$

其中：

$$
M_{i,j}
=
\begin{cases}
0,&j\le i\\
-\infty,&j>i.
\end{cases}
$$

再：

$$
A
=
\operatorname{softmax}(S_{masked},\ axis=-1),
$$

最后：

$$
\boxed{O=AV}.
$$

所以这节实现没有增加新的 Attention 数学；它只是第一次把 Step 11–12.3 的全部组件按照正确顺序组织成一个可复用 forward。


In [ ]:
# Example usage only. This notebook master remains unexecuted.
B, T, C = 2, 4, 6
head_dim = 3

head = CausalSelfAttentionHead(
    input_dims=C,
    head_dim=head_dim,
)

x = mx.zeros((B, T, C))
out = head(x)

print('input shape :', x.shape)
print('output shape:', out.shape)
print('expected    :', (B, T, head_dim))

# Only the Q/K/V Linear arrays should be trainable here.
# The causal mask is created locally inside __call__ and is not stored on head.
print('trainable parameter tree:')
print(head.trainable_parameters())


## 5. 如果把 Mask 存进 Module，为什么必须显式 Freeze？

本节主实现没有把 mask 放进 `self.*`。

但理解 MLX module state 很重要。

MLX 0.32.2 的 `Module` 规则是：module 中的 `mx.array` members 会被 `parameters()` 收集，arrays 默认 trainable，除非被 `freeze`。

所以如果未来为了避免重复构造 mask，设计类似：

```python
self.causal_mask = cached_mask
```

概念上应该再明确：

```python
self.freeze(
    recurse=False,
    keys='causal_mask',
    strict=True,
)
```

这样它仍然可以作为 module state 被保存/访问，但不会出现在 `trainable_parameters()` 里参与梯度更新。

不过缓存还要额外考虑：

```text
max context length
当前 T 的 slice
dtype
device / memory
```

这些属于后续工程优化，不是 Step 12.4 的目标。

当前版本采用 forward-local mask，优点是概念简单、不会误注册为 trainable array，而且自然适配当前 T。


## 6. 这个 Head 已经做到了什么，又还没做到什么？

现在这个 module 已经是真正的 **single causal self-attention head**：

```text
input X
↓
Q/K/V
↓
scaled compatibility
↓
causal masking
↓
stable softmax
↓
Value retrieval
↓
causal context-dependent output
```

并且每个 Query position `i` 的输出只能依赖：

$$
v_0,\ldots,v_i.
$$

不会读取 future Values。

但它仍然故意没有：

- Multi-Head Attention；
- heads concatenation；
- output projection；
- residual connection；
- normalization；
- feed-forward block；
- 专门打印/分析 attention matrix。

这些不是本节内容。

尤其 Step 12.5 才会专门 inspect 一个 tiny `T×T` attention matrix，验证未来概率为零；本轮不提前做那项检查。


## 7. Step 12.4 材料边界

- single causal head 的 trainable arrays 只有 Q/K/V linear projections；`1/sqrt(d_h)` 和 causal visibility rule 都是结构常量。
- 本节主实现根据当前 T 在 forward 内局部构造 causal mask，因此它不进入 MLX module parameter tree。
- MLX 0.32.2 若把 `mx.array` 存为 module attribute，会被 `parameters()` 收集且默认 trainable；缓存固定 mask 时应显式 freeze，使其成为 non-trainable state。
- forward shape：`X:(B,T,C) -> Q/K/V:(B,T,d_h) -> scores:(B,T,T) -> weights:(B,T,T) -> O:(B,T,d_h)`。
- mask 必须在 softmax 前作用到 scaled scores；softmax 仍沿 Key axis `axis=-1`。
- `mx.softmax` 直接承担 numerically stable softmax；本节不手写 stable-softmax kernel。
- 本轮只准备 Step 12.4；不进入 Step 12.5，不做专门 attention-matrix inspection，也不进入 Multi-Head Attention。
